# Activity: Postman WebSocket Echo Server 

In this activity, we will connect to the Postman WebSocket echo service to test basic WebSocket client functionality by sending structured JSON payloads and observing the echoed responses.

__Why are we looking at this?__ Echo servers provide a simple way to verify WebSocket connections work correctly before moving to complex real-time APIs. By sending data and receiving it back unchanged, you can confirm your client handles the WebSocket lifecycle (connect, send, receive, close) without needing authentication or domain-specific logic.

> __Learning Objectives__
> 
> By the end of this activity, you will be able to:
> * __Configure a WebSocket client:__ Establish a connection to a WebSocket endpoint and send JSON payloads using the `HTTP.WebSockets` module in Julia.
> * __Handle message formats:__ Normalize incoming WebSocket frames (which can arrive as strings or byte arrays) and parse JSON responses safely with error handling.
> * __Manage connection lifecycle:__ Control the message receive loop with counters and close the WebSocket connection cleanly after collecting data.

Let's get started!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

Let's set up our code environment:


In [1]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

### Constants
We define the Postman WebSocket URL, the message to echo, and a small limit on how many responses to capture before closing the connection.


In [ ]:
url = "wss://ws.postman-echo.com/raw"; # WebSocket URL for Postman Echo
message_text = "Hello from Julia!"; # Text to send to the echo server
message_text_reversed = reverse(message_text); # reverse before sending so we can flip it back on receipt
number_of_messages = 1; # Number of messages (only 1)
const WS = HTTP.WebSockets; # Create alias for WebSockets module for convenience


### Helper Implementation
To understand the low-level structure of WebSocket frames, we implement a helper function that computes the frame header components for a given message.

> __Computing frame components__
> 
> The `frame_components(...)` function takes a message string and a `masked` flag, then returns a `Dict` containing all the WebSocket frame header fields:
> * **FIN bit:** Always 1 for single-frame messages (indicates this is the final fragment)
> * **Opcode:** Set to 0x1 for text data (vs. 0x2 for binary)
> * **Mask bit:** 1 for client-to-server frames, 0 for server-to-client frames
> * **Payload length fields:** The actual message size and how it's encoded in the header

The function determines how to encode the payload length:

> __Length field encoding__
>
> Messages under 126 bytes use the 7-bit length field directly. Messages up to 65,535 bytes add a 2-byte extended length field (and set the 7-bit field to 126). Larger messages add an 8-byte extended length field (and set the 7-bit field to 127).

Finally, the function computes the total header size:

> __Header size calculation__
>
> The total header size equals the base header (2 bytes) plus any extended length bytes (0, 2, or 8) plus the masking key (4 bytes if masked, 0 otherwise). This tells us exactly how many bytes WebSocket uses for metadata before the actual payload.

This helper function allows us to inspect frame structure before and after transmission.

In [ ]:
frame_components(message::AbstractString; masked::Bool) = Dict(
    
    "fin" => 1,
    "rsv1-3" => 0,
    "opcode" => "text (0x1)",
    "mask" => masked ? 1 : 0,
    
    "payload_length_field" => begin
        L = ncodeunits(message)
        L < 126 ? L : (L <= typemax(UInt16) ? 126 : 127)
    end,
    "actual_payload_length" => ncodeunits(message),
    "extended_length_bytes" => begin
        L = ncodeunits(message)
        L < 126 ? 0 : (L <= typemax(UInt16) ? 2 : 8)
    end,
    "total_header_bytes" => begin
        L = ncodeunits(message)
        base = L < 126 ? 2 : (L <= typemax(UInt16) ? 4 : 10)
        base + (masked ? 4 : 0)
    end,
);

___
## Task 1: Prepare the Echo Payload
The Postman echo server returns whatever we send without modification. We will construct a JSON payload with a reversed message string to test both structured data transmission and content restoration.

> __Creating a JSON payload structure__
>
> We use `JSON.json(...)` to serialize a Julia `Dict` into a JSON string. The dictionary contains two fields: 
> * **event:** Set to `"echo"` to indicate the type of request we are making to the server (many WebSocket APIs use an event field to route different message types)
> * **message:** Contains our reversed text (`message_text_reversed`), which is the actual data payload we want the server to echo back to us

This reversal strategy serves a specific verification purpose:

> __Why reverse the message?__
>
> By reversing the message before sending, we can verify the echo works correctly by reversing it again upon receipt. If the restored message matches the original `message_text`, we confirm that both the JSON structure and the message content survived the round trip unchanged.

In [3]:
payload = JSON.json(Dict(
    "event" => "echo",
    "message" => message_text_reversed,
)); # JSON payload to send (note: message is reversed on purpose)

payload


"{\"event\":\"echo\",\"message\":\"!ailuJ morf olleH\"}"

___
## Task 2: Implement the WebSocket Echo Loop
With our payload ready, we can open a WebSocket connection to the Postman echo service, send the message, and process the response while analyzing the frame structure.

Before opening the connection, we analyze the outgoing frame:

> __Analyzing the outgoing frame__
>
> We call `frame_components(payload; masked=true)` to compute the expected header fields for our outgoing message. Client-to-server frames must be masked, so we set `masked=true`. The function returns a dictionary showing the FIN bit, opcode, mask bit, payload length encoding, and total header size. Logging this information helps us understand how WebSocket frames package our JSON payload before transmission.

With frame analysis in place, we can establish the connection:

> __Opening the connection and sending data__
>
> [The `WS.open(...)` function](https://juliaweb.github.io/HTTP.jl/stable/reference/#HTTP.open) establishes a WebSocket connection to the specified URL. The `do` block receives a `ws` handle for the connection. We immediately send our JSON payload using `WS.send(ws, payload)`, which transmits the message to the server inside a masked WebSocket frame.

Once the message is sent, we need to handle the incoming response:

> __Normalizing incoming frames__
>
> WebSocket frames can arrive as either a `String` or `Vector{UInt8}` (byte array). The line `s = raw isa AbstractString ? raw : String(raw)` normalizes both formats to a string. We then call `frame_components(s; masked=false)` to analyze the incoming frame structure (server-to-client frames are never masked). This shows us how the server packaged the echoed response.

After analyzing the frame, we parse and restore the message content:

> __Parsing and restoring the message__
>
> We attempt to parse the response as JSON using a `try-catch` block. If parsing succeeds, we extract the `"message"` field and reverse it using `reverse(String(parsed["message"]))` to restore the original text. If JSON parsing fails, we treat the entire response as plain text and reverse it directly. The `restored_message` variable holds the final result, which should match the original `message_text` if the echo worked correctly.

Finally, we manage the connection lifecycle:

> __Controlling the receive loop__
>
> The `seen` counter tracks how many messages we have received. After processing each message, we increment `seen` and check if we have reached `max_messages`. When the limit is reached, we call `WS.close(ws)` to gracefully terminate the connection and exit the loop. This pattern ensures deterministic cleanup and prevents the connection from remaining open indefinitely.

In [4]:


outgoing_frame = frame_components(payload; masked=true);
@info "outgoing frame header preview" outgoing_frame;

WS.open(url) do ws
    WS.send(ws, payload) # send payload once when the socket opens
    seen = 0; # count of messages seen
    max_messages = number_of_messages; # number of messages to receive (then we close the connection)

    for raw in ws
        s = raw isa AbstractString ? raw : String(raw) # raw can be String or Vector{UInt8}; normalize to String

        incoming_frame = frame_components(s; masked=false)

        parsed = try
            JSON.parse(s)
        catch err
            nothing # not valid JSON; treat as plain text
        end

        if parsed !== nothing
            restored_message = haskey(parsed, "message") ? reverse(String(parsed["message"])) : nothing
            @info "json echo" parsed incoming_frame=incoming_frame restored_message=restored_message
        else
            restored_message = reverse(s)
            @info "text echo" message=s incoming_frame=incoming_frame restored_message=restored_message
        end

        seen += 1 # update message count
        if seen >= max_messages
            @info "max message count reached; closing websocket" count=seen
            WS.close(ws) # close the WebSocket connection
            break
        end
    end
end


┌ Info: outgoing frame header preview
│   outgoing_frame = Dict{String, Any}("fin" => 1, "mask" => 1, "total_header_bytes" => 6, "extended_length_bytes" => 0, "actual_payload_length" => 46, "rsv1-3" => 0, "opcode" => "text (0x1)", "payload_length_field" => 46)
└ @ Main /Users/jdv27/Desktop/julia_work/CHEME-140-eCornell-Repository/courses/CHEME-142/module-3/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X20sZmlsZQ==.jl:23
┌ Info: json echo
│   parsed = JSON.Object{String, Any}("event" => "echo", "message" => "!ailuJ morf olleH")
│   incoming_frame = Dict{String, Any}("fin" => 1, "mask" => 0, "total_header_bytes" => 2, "extended_length_bytes" => 0, "actual_payload_length" => 46, "rsv1-3" => 0, "opcode" => "text (0x1)", "payload_length_field" => 46)
│   restored_message = Hello from Julia!
└ @ Main /Users/jdv27/Desktop/julia_work/CHEME-140-eCornell-Repository/courses/CHEME-142/module-3/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X20sZmlsZQ==.jl:43
┌ Info: max message count reache

HTTP.Messages.Response:
"""
HTTP/1.1 101 Switching Protocols
Date: Wed, 17 Dec 2025 19:58:54 GMT
Connection: upgrade
Server: nginx
Upgrade: websocket
Sec-WebSocket-Accept: EQEBZxrAAoW7+iIKKdth0Rx2kbc=

"""

___
## Summary
This activity demonstrated how to establish a WebSocket connection to an echo service, send structured JSON data, analyze frame headers, and restore the echoed message content.

> __Key Takeaways__
> 
> * **WebSocket frame structure:** Client-to-server frames use masking and contain header fields (FIN, opcode, mask bit, payload length) that can be computed from the message size to understand how data is packaged for transmission.
> * **Message normalization and restoration:** WebSocket frames arrive as strings or byte arrays, requiring normalization, defensive JSON parsing, and content restoration (such as reversing text) to verify that data survived the round trip unchanged.
> * **Connection lifecycle control:** Track message counts and call `close` explicitly to terminate connections cleanly after collecting the desired data, ensuring proper resource management.

Echo servers provide a safe testing environment for WebSocket client code and frame analysis before integrating with production real-time APIs.
___